In [ ]:
import sys
from pathlib import Path
import tempfile, py7zr
import geopandas as gpd
import matplotlib.pyplot as plt
import glob, json, tempfile
from pathlib import Path
import py7zr
import geopandas as gpd
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
import numpy as np
from matplotlib.colors import ListedColormap, Normalize, TwoSlopeNorm


# Find the repo root (folder containing "functions/") and add it + its parent to sys.path
repo = Path.cwd()
while not (repo / "functions").exists() and repo.parent != repo:
    repo = repo.parent
sys.path[:0] = [str(repo), str(Path.cwd().parent.parent)]

from comap_constants import *
from comap_plotting import plot_comap
from functions.plot_hillshade import plot_hillshade


from utils import *

### PLOT BASE COMPA MAP LAYER ###

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    py7zr.SevenZipFile(COMPAP_PATH).extractall(tmp)
    shp = next(Path(tmp).rglob("*.shp"))
    gdf = gpd.read_file(shp).set_crs("EPSG:26913", allow_override=True)

# Hillshade base
fig, ax = plot_hillshade()

# Counties on top
plt_counties(COUNTIES_PATH, county_edgecolor="gray", county_linewidth=1.0, ax=ax)

# Full styled COMaP on top
plot_comap(gdf, ax=ax, title="COMaP Conservation Status")

plt.tight_layout()
plt.show()

### PLOT SINGLE COMAP LEVEL WITH GEOTIFS ###

In [ ]:
LAYERS_TO_PLOT = ["5 day max precip"]

# LAYERS_TO_PLOT = [
#     "drought__D1",
#     "drought__D2",
#     "drought__D3",
#     "drought__D4",
#     "snowfall",
#     "Days Exceeding 95th Percentile Maximum Temperature",
#     "Change in Number of Frost Days",
#     "Rx1day",
#     "5 day max precip",
# ]

LEVEL = "V"
TARGET_CRS = "EPSG:3857"

OUT_DIR = Path("/Users/kylabazlen/Documents/Climate_Roadmap/outputs/cs_plots")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CBAR_JSON = Path("/Users/kylabazlen/Documents/Climate_Roadmap/clim_data_lcc/custom_colormaps.json")
INDEX_PAH = Path("/Users/kylabazlen/Documents/Climate_Roadmap/clim_data_lcc/geotiff_index_20260506.csv")
cbar_spec = json.loads(CBAR_JSON.read_text())["variables"]


selected = LAYERS_TO_PLOT or list(layers.keys())

# Load COMaP once and subset for plotting
with tempfile.TemporaryDirectory() as tmp_comap:
    py7zr.SevenZipFile(COMPAP_PATH).extractall(tmp_comap)
    shp = next(Path(tmp_comap).rglob("*.shp"))
    gdf = gpd.read_file(shp).set_crs("EPSG:26913", allow_override=True).to_crs(TARGET_CRS)
subset = gdf[gdf["FinalModSt"] == LEVEL]

for layer_name in selected:
    if layer_name not in layers:
        print(f"⚠️  Unknown layer: {layer_name}")
        continue
    cfg = layers[layer_name]
    files = sorted(glob.glob(cfg["path"]))
    if not files:
        print(f"⚠️  No files for {layer_name}: {cfg['path']}")
        continue

    for fp in files:
        cmap, norm, labels = build_cmap_norm(cfg, cbar_spec, raster_path=fp)        
        arr, extent = reproject_raster(fp)

        # Create per-iteration output folder
        iter_dir = OUT_DIR / Path(fp).stem
        iter_dir.mkdir(parents=True, exist_ok=True)

        # Hillshade base (no colorbar on this figure)
        fig, ax = plot_hillshade(target_crs=TARGET_CRS, topo_alpha=1, zfactor=5)

        # Climate raster (zorder=2)
        im = ax.imshow(arr, extent=extent, cmap=cmap, norm=norm, alpha=0.7, zorder=2)

        # County boundaries on top of raster (zorder=3)
        plt_counties(COUNTIES_PATH, county_edgecolor="black", county_linewidth=1.0, ax=ax)

        # COMaP outlines on top of everything (zorder=4)
        subset.plot(ax=ax, facecolor="none", edgecolor="red", linewidth=0.75, zorder=4)

        # Lock axis limits to the climate raster bounds (Colorado)
        ax.set_xlim(extent[0], extent[1])
        ax.set_ylim(extent[2], extent[3])
        ax.set_title(f"{cfg.get('label', layer_name)} — COMaP {FINALMODST_LABELS[LEVEL]}")
        ax.set_axis_off()

        # Save main map (without colorbar)
        plot_path = iter_dir / f"{Path(fp).stem}.png"
        fig.tight_layout()
        fig.savefig(plot_path, bbox_inches="tight")
        plt.close(fig)

        # Save colorbar separately using a ScalarMappable (horizontal)
        from matplotlib.cm import ScalarMappable
        sm = ScalarMappable(norm=norm, cmap=cmap)
        sm.set_array([])
        fig_cb, ax_cb = plt.subplots(figsize=(6, 0.7))
        cbar = fig_cb.colorbar(sm, cax=ax_cb, orientation='horizontal')
        # Remove tick markers (lines) and outline, keep labels floating below
        cbar.ax.tick_params(length=0, pad=6)
        cbar.outline.set_visible(False)
        # Optional: increase label fontsize
        cbar.ax.xaxis.set_tick_params(labelsize=9)
        cbar.set_label(cfg.get("cbar_label", ""))
        fig_cb.subplots_adjust(left=0.05, right=0.95, top=0.85, bottom=0.25)
        cbar_path = iter_dir / f"{Path(fp).stem}__colorbar.png"
        fig_cb.savefig(cbar_path, bbox_inches='tight', dpi=150)
        plt.close(fig_cb)

        print(f"✓ saved plot: {plot_path}")
        print(f"✓ saved colorbar: {cbar_path}")


KeyError: 'variables'